In [ ]:
import duckdb
import pandas as pd

# Kết nối DuckDB (chế độ in-memory để xử lý nhanh)
con = duckdb.connect(database=':memory:')

# Khai báo danh sách các file CSV để quản lý tập trung
files = {
    'products': '../dataset/products.csv',
    'orders': '../dataset/orders.csv',
    'customers': '../dataset/customers.csv',
    'returns': '../dataset/returns.csv',
    'web_traffic': '../dataset/web_traffic.csv'
}

print("Đã thiết lập kết nối DuckDB và danh sách tệp tin.")

Đã thiết lập kết nối DuckDB và danh sách tệp tin.


In [4]:
print("--- BẮT ĐẦU KIỂM TRA DỮ LIỆU (DATA AUDIT) ---")

for table, file in files.items():
    print(f"\n[Kiểm tra bảng: {table.upper()}]")
    
    # 1. Kiểm tra sự tồn tại của file và cấu trúc cột
    try:
        columns = con.execute(f"DESCRIBE SELECT * FROM '{file}'").df()['column_name'].tolist()
        print(f"- Cấu trúc cột: {columns}")
    except Exception as e:
        print(f"- LỖI: Không tìm thấy file hoặc file lỗi: {e}")
        continue
    
    # 2. Kiểm tra Duplicate Khóa chính
    # Tự động xác định PK dựa trên tên bảng
    pk = f"{table[:-1]}_id" if table != 'orders' else 'order_id'
    if table == 'returns': pk = 'return_id'
    
    try:
        dup_count = con.execute(f"SELECT COUNT({pk}) - COUNT(DISTINCT {pk}) FROM '{file}'").fetchone()[0]
        print(f"- Số lượng trùng lặp khóa chính ({pk}): {dup_count}") [cite: 71]
    except:
        print(f"- Lưu ý: Không kiểm tra được PK cho bảng này.")

    # 3. Kiểm tra logic nghiệp vụ (Business Logic)
    if table == 'products':
        # Kiểm tra cogs < price theo yêu cầu đề bài [cite: 283]
        invalid_logic = con.execute(f"SELECT COUNT(*) FROM '{file}' WHERE cogs >= price").fetchone()[0]
        print(f"- Số dòng vi phạm logic (cogs >= price): {invalid_logic}")
        
    if table == 'orders':
        # Kiểm tra định dạng ngày tháng [cite: 72]
        invalid_dates = con.execute(f"SELECT COUNT(*) FROM '{file}' WHERE try_cast(order_date as DATE) IS NULL").fetchone()[0]
        print(f"- Số dòng lỗi định dạng order_date: {invalid_dates}")
        
    if table == 'web_traffic':
        # Kiểm tra bounce_rate hợp lệ (0-1)
        invalid_bounce = con.execute(f"SELECT COUNT(*) FROM '{file}' WHERE bounce_rate < 0 OR bounce_rate > 1").fetchone()[0]
        print(f"- Số dòng có bounce_rate không hợp lệ: {invalid_bounce}")

print("\n--- HOÀN TẤT KIỂM TRA ---")

--- BẮT ĐẦU KIỂM TRA DỮ LIỆU (DATA AUDIT) ---

[Kiểm tra bảng: PRODUCTS]
- Cấu trúc cột: ['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']
- Số lượng trùng lặp khóa chính (product_id): 0
- Lưu ý: Không kiểm tra được PK cho bảng này.
- Số dòng vi phạm logic (cogs >= price): 0

[Kiểm tra bảng: ORDERS]
- Cấu trúc cột: ['order_id', 'order_date', 'customer_id', 'zip', 'order_status', 'payment_method', 'device_type', 'order_source']
- Số lượng trùng lặp khóa chính (order_id): 0
- Lưu ý: Không kiểm tra được PK cho bảng này.
- Số dòng lỗi định dạng order_date: 0

[Kiểm tra bảng: CUSTOMERS]
- Cấu trúc cột: ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel']
- Số lượng trùng lặp khóa chính (customer_id): 0
- Lưu ý: Không kiểm tra được PK cho bảng này.

[Kiểm tra bảng: RETURNS]
- Cấu trúc cột: ['return_id', 'order_id', 'product_id', 'return_date', 'return_reason', 'return_quantity', 'refund_amount']
- Số lượng trùng lặp

In [ ]:
# Q1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần 
# mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?

query_q1 = """
WITH RawOrders AS (
    SELECT 
        customer_id, 
        order_date,
        LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) as prev_date
    FROM '../dataset/orders.csv'
),
Gaps AS (
    SELECT 
        date_diff('day', prev_date, order_date) as gap
    FROM RawOrders
    WHERE prev_date IS NOT NULL
)
SELECT MEDIAN(gap) FROM Gaps;
"""

result = con.execute(query_q1).fetchone()[0]
print(f"Kết quả: {result} ngày")

Kết quả: 144.0 ngày


In [ ]:
# Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
# trung bình cao nhất, với công thức (price − cogs)/price?

query_q2 = """
SELECT 
    segment, 
    AVG((price - cogs) / price) AS avg_gross_margin
FROM read_csv_auto('../dataset/products.csv')
GROUP BY segment
ORDER BY avg_gross_margin DESC
LIMIT 1;
"""

result_q2 = con.execute(query_q2).fetchone()
print(f"Phân khúc có lợi nhuận cao nhất: {result_q2[0]}")
print(f"Giá trị trung bình: {result_q2[1]}")

Phân khúc có lợi nhuận cao nhất: Standard
Giá trị trung bình: 0.3134417484388478


In [ ]:
# Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
# returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

query_q3 = """
SELECT 
    r.return_reason, 
    COUNT(*) AS reason_count
FROM read_csv_auto('../dataset/returns.csv') r
JOIN read_csv_auto('../dataset/products.csv') p 
    ON r.product_id = p.product_id
WHERE p.category = 'Streetwear'
GROUP BY r.return_reason
ORDER BY reason_count DESC
LIMIT 1;
"""

result_q3 = con.execute(query_q3).fetchone()
print(f"Lý do trả hàng phổ biến nhất: {result_q3[0]}")

Lý do trả hàng phổ biến nhất: wrong_size


In [ ]:
# Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
# bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

query_q4 = """
SELECT 
    traffic_source, 
    AVG(bounce_rate) as avg_bounce
FROM read_csv_auto('../dataset/web_traffic.csv')
GROUP BY traffic_source
ORDER BY avg_bounce ASC
LIMIT 1;
"""
res_q4 = con.execute(query_q4).fetchone()
print(f"Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: {res_q4[0]} (Bounce Rate TB: {res_q4[1]:.4f})")

Đáp án Q4: email_campaign (Bounce Rate TB: 0.0045)


In [ ]:
# Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
# không null) xấp xỉ là bao nhiêu?

query_q5 = """
SELECT 
    (COUNT(promo_id) * 100.0 / COUNT(*)) AS promo_percentage
FROM read_csv_auto('../dataset/order_items.csv');
"""
res_q5 = con.execute(query_q5).fetchone()[0]
print(f"Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi: {res_q5:.2f}%")

Đáp án Q5: 38.66%


In [ ]:
# Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
# đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
# nhóm)

query_q6 = """
SELECT 
    c.age_group,
    COUNT(o.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) AS avg_orders_per_customer
FROM read_csv_auto('../dataset/customers.csv') c
JOIN read_csv_auto('../dataset/orders.csv') o
    ON c.customer_id = o.customer_id
WHERE c.age_group IS NOT NULL
GROUP BY c.age_group
ORDER BY avg_orders_per_customer DESC
LIMIT 1;
"""
res_q6 = con.execute(query_q6).fetchone()
print(f"Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: Nhóm tuổi {res_q6[0]} (Trung bình: {res_q6[1]:.2f} đơn/người)")

Đáp án Q6: Nhóm tuổi 55+ (Trung bình: 7.27 đơn/người)


In [ ]:
# Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
# sales_train.csv?

query_q7 = """
SELECT 
    g.region,
    SUM(oi.quantity * oi.unit_price - COALESCE(oi.discount_amount, 0)) AS total_revenue
FROM read_csv_auto('../dataset/orders.csv') o
JOIN read_csv_auto('../dataset/order_items.csv') oi 
    ON o.order_id = oi.order_id
JOIN read_csv_auto('../dataset/geography.csv') g 
    ON o.zip = g.zip
WHERE o.order_status != 'cancelled'
GROUP BY g.region
ORDER BY total_revenue DESC
LIMIT 1;
"""
res_q7 = con.execute(query_q7).fetchone()
print(f"Vùng tạo ra tổng doanh thu cao nhất: Vùng {res_q7[0]} (Doanh thu: {res_q7[1]:.2f})")

Đáp án Q7: Vùng East (Doanh thu: 6617595804.01)


In [ ]:
# Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
# thanh toán nào được sử dụng nhiều nhất?
query_q8 = """
SELECT 
    payment_method, 
    COUNT(*) AS cancel_count
FROM read_csv_auto('../dataset/orders.csv')
WHERE order_status = 'cancelled'
GROUP BY payment_method
ORDER BY cancel_count DESC
LIMIT 1;
"""
res_q8 = con.execute(query_q8).fetchone()
print(f"Phương thức thanh toán bị huỷ đơn nhiều nhất: {res_q8[0]} (Số đơn hủy: {res_q8[1]})")

Đáp án Q8: credit_card (Số đơn hủy: 28452)


In [ ]:
# Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
# nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
# với products theo product_id)?

query_q9 = """
WITH ReturnCounts AS (
    SELECT p.size, COUNT(*) as return_count
    FROM read_csv_auto('../dataset/returns.csv') r
    JOIN read_csv_auto('../dataset/products.csv') p 
        ON r.product_id = p.product_id
    WHERE p.size IN ('S', 'M', 'L', 'XL')
    GROUP BY p.size
),
OrderCounts AS (
    SELECT p.size, COUNT(*) as order_count
    FROM read_csv_auto('../dataset/order_items.csv') oi
    JOIN read_csv_auto('../dataset/products.csv') p 
        ON oi.product_id = p.product_id
    WHERE p.size IN ('S', 'M', 'L', 'XL')
    GROUP BY p.size
)
SELECT 
    o.size,
    (r.return_count * 1.0 / o.order_count) as return_rate
FROM OrderCounts o
JOIN ReturnCounts r ON o.size = r.size
ORDER BY return_rate DESC
LIMIT 1;
"""
res_q9 = con.execute(query_q9).fetchone()
print(f"Kích thước có tỷ lệ trả hàng cao nhất: Size {res_q9[0]} (Tỷ lệ trả hàng: {res_q9[1]:.4f})")

Đáp án Q9: Size S (Tỷ lệ trả hàng: 0.0565)


In [ ]:
# Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
# mỗi đơn hàng cao nhất?

query_q10 = """
SELECT 
    installments, 
    AVG(payment_value) as avg_payment
FROM read_csv_auto('../dataset/payments.csv')
GROUP BY installments
ORDER BY avg_payment DESC
LIMIT 1;
"""
res_q10 = con.execute(query_q10).fetchone()
print(f"Kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn cao nhất: {res_q10[0]} kỳ (Giá trị TB: {res_q10[1]:.2f})")

Đáp án Q10: 6 kỳ (Giá trị TB: 24446.65)
